In [1]:
import os
import torch
from datasets import load_dataset, DatasetDict, Audio
import librosa
import numpy as np
import evaluate


/opt/conda/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check GPU availability
import torch

print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not detected. Model will be trained on the CPU.")

GPU Available: True
Using GPU: Tesla T4


In [14]:
torch.cuda.empty_cache()

In [3]:
dataset = load_dataset("assoni2002/jailbreak_classification_without_changing_freq")

Generating train split: 100%|██████████| 2040/2040 [00:04<00:00, 453.54 examples/s]


In [7]:
data['train'][0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7fd2ee3a7280>,
 'labels': 0}

In [8]:
dataset = data.cast_column("audio", Audio())

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 2040
    })
})

In [ ]:
# Extract decoded audio arrays from the dataset
audio_arrays = [np.array(example["audio"]["array"]) for example in dataset['train']]

# Compare audio arrays to find duplicates
unique_audio_indices = {}
duplicates = set()

for index, audio in enumerate(audio_arrays):
    audio_tuple = tuple(audio)  # Convert array to a hashable type
    if audio_tuple in unique_audio_indices:
        duplicates.add(index)
    else:
        unique_audio_indices[audio_tuple] = index

# Select only unique audio indices
filtered_dataset = dataset['train'].select([i for i in range(len(dataset['train'])) if i not in duplicates])

print("Original dataset size:", len(dataset['train']))
print("Filtered dataset size:", len(filtered_dataset))

Original dataset size: 4886
Filtered dataset size: 4886


In [ ]:
filtered_dataset[0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x1756857b0>,
 'labels': 0}

In [8]:
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")

/opt/conda/envs/myenv/lib/python3.10/site-packages/transformers/configuration_utils.py:309: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [ ]:
# Find the 5 longest audio samples
def find_top_longest_audios(dataset, top_n=5):
    # Create a list of (index, length) for each audio sample
    audio_lengths = [(i, len(example["audio"]["array"])) for i, example in enumerate(dataset)]

    # Sort by length in descending order
    sorted_audio_lengths = sorted(audio_lengths, key=lambda x: x[1], reverse=True)
    
    # Get the top `top_n` audios
    top_longest = sorted_audio_lengths[:top_n]
    return top_longest

# Find the 5 longest audios in the filtered dataset
top_longest_audios = find_top_longest_audios(filtered_dataset, top_n=5)

# Print results
print("--- 5 Longest Audio Samples ---")
for idx, length in top_longest_audios:
    print(f"Audio Index: {idx}, Length (samples): {length}")

--- 5 Longest Audio Samples ---
Audio Index: 3140, Length (samples): 578944
Audio Index: 4147, Length (samples): 566784
Audio Index: 2498, Length (samples): 564337
Audio Index: 3896, Length (samples): 503424
Audio Index: 980, Length (samples): 502848


In [ ]:
from IPython.display import Audio

# Retrieve the longest audio sample by its index
index = 3140  # Replace with the desired index from the "Top 5 Longest Audio Samples"
longest_audio_sample = filtered_dataset[index]["audio"]

# Get the raw waveform array and sampling rate
audio_array = longest_audio_sample["array"]
sampling_rate = longest_audio_sample["sampling_rate"]

# Play the audio
Audio(data=audio_array, rate=sampling_rate)

In [ ]:
def find_max_sampling_rate(dataset):
    # Extract all sampling rates from the "audio" column
    sampling_rates = [example["audio"]["sampling_rate"] for example in dataset]

    # Find and return the maximum sampling rate
    max_rate = max(sampling_rates)
    return max_rate

# Check the maximum sampling rate in the dataset
max_rate = find_max_sampling_rate(filtered_dataset)  

print(f"The maximum sampling rate in the dataset is: {max_rate} Hz")

In [ ]:
from IPython.display import Audio

# Retrieve the longest audio sample by its index
index = 3140  # Replace with the desired index from the "Top 5 Longest Audio Samples"
longest_audio_sample = filtered_dataset[index]["audio"]

# Get the raw waveform array and sampling rate
audio_array = longest_audio_sample["array"]
sampling_rate = longest_audio_sample["sampling_rate"]

# Play the audio
Audio(data=audio_array, rate=sampling_rate)

In [ ]:
def hear_difference(audio_array, original_sampling_rate, target_sampling_rate=16000):
    # Play original audio
    print("Playing original audio...")
    original_audio = Audio(data=audio_array, rate=original_sampling_rate)
    display(original_audio)
    
    # Resample audio to the target sampling rate (16000 Hz)
    print("\nResampling audio to 16,000 Hz...")
    resampled_audio = librosa.resample(audio_array, orig_sr=original_sampling_rate, target_sr=target_sampling_rate)
    
    # Play resampled audio
    resampled_audio_player = Audio(data=resampled_audio, rate=target_sampling_rate)
    display(resampled_audio_player)

    # Return both original and resampled audio arrays for further analysis if needed
    return original_audio, resampled_audio

# Retrieve an audio sample from the dataset
index = 3140  # Replace with the index of the audio you want to hear
audio_sample = filtered_dataset[index]["audio"]

# Get the raw waveform and sampling rate
audio_array = audio_sample["array"]
original_sampling_rate = audio_sample["sampling_rate"]

# Call the function to hear both audio versions
hear_difference(audio_array, original_sampling_rate)

In [ ]:
import matplotlib.pyplot as plt

resampled_audio = librosa.resample(audio_array, orig_sr=original_sampling_rate, target_sr=16000)
# Plot original waveform
plt.figure(figsize=(12, 6))

plt.subplot(2, 1, 1)
plt.plot(audio_array[:5000])  # Plot the first 5000 samples
plt.title("Original Audio Waveform (44,100 Hz)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")

# Plot resampled waveform
plt.subplot(2, 1, 2)
plt.plot(resampled_audio[:2000])  # Plot the first 2000 samples (resampled to fewer samples)
plt.title("Resampled Audio Waveform (16,000 Hz)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()

In [ ]:
import librosa.display
# Compute spectrogram for original and resampled audio
original_spectrogram = librosa.amplitude_to_db(
    np.abs(librosa.stft(audio_array)), ref=np.max
)
resampled_spectrogram = librosa.amplitude_to_db(
    np.abs(librosa.stft(resampled_audio)), ref=np.max
)

# Plot spectrogram of the original audio
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
librosa.display.specshow(
    original_spectrogram, sr=original_sampling_rate, x_axis="time", y_axis="hz"
)
plt.colorbar(format="%+2.0f dB")
plt.title("Spectrogram - Original Audio (44,100 Hz)")

# Plot spectrogram of the resampled audio
plt.subplot(2, 1, 2)
librosa.display.specshow(
    resampled_spectrogram, sr=16000, x_axis="time", y_axis="hz"
)
plt.colorbar(format="%+2.0f dB")
plt.title("Spectrogram - Resampled Audio (16,000 Hz)")

plt.tight_layout()
plt.show()

In [5]:
def preprocess_function(examples):
    # Extract raw audio arrays directly from the dataset
    audio_arrays = [example["array"] for example in examples["audio"]]
    
    # Preprocess the audio arrays using the feature extractor while keeping the original sampling rate
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,  # Use the original sampling rate of the audio
        padding=True,  # Pad shorter audio to ensure batch uniformity
        truncation=False  # Do not truncate; process full audio
    )
    
    # Attach classification labels
    inputs["labels"] = examples["labels"]
    
    return inputs

In [10]:
# Perform an 80-20 train-test split directly on the filtered dataset
data_split = dataset['train'].train_test_split(
    test_size=0.2,  # Define 20% as test set
    seed=42         # Ensure reproducibility
)

# Print the resulting splits
print("--- Train and Test Dataset Splits ---")
print(data_split)

--- Train and Test Dataset Splits ---
DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 1632
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 408
    })
})


In [11]:
# Apply preprocessing to the dataset
processed_dataset = data_split.map(preprocess_function, remove_columns="audio", batched=True)

Map:   0%|          | 0/1632 [00:00<?, ? examples/s]

: 

In [ ]:
accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer

num_labels = 2
model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base", num_labels=num_labels)

/opt/conda/envs/myenv/lib/python3.10/site-packages/transformers/configuration_utils.py:309: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(
Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
training_args = TrainingArguments(
    output_dir="Wav2Vec2_jailbreak_classification",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
)                                      #model.save_pretrained(path)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.85 GiB. GPU 0 has a total capacity of 14.75 GiB of which 2.36 GiB is free. Including non-PyTorch memory, this process has 12.38 GiB memory in use. Of the allocated memory 10.29 GiB is allocated by PyTorch, and 1.96 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
final_results = trainer.evaluate(eval_dataset=encoded_ds["test"])

print("\nFinal Test Set Performance:")
print(final_results)

In [ ]:

trainer.push_to_hub()
